# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/daniyalhaider236/flyrank-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*



**Task type:** Lane 2 is a ranking/scoring problem (established in Week 2), evaluated at Precision@20 /
Precision@50. That means I need a model that outputs a *score I can rank by*, not just a label — so per
the toolkit's "which first?" guidance, I use a classifier's `predict_proba` and rank by it, rather than
its hard 0/1 prediction.

I compare two methods, cheapest first: **Logistic Regression** (readable, a coefficient per feature I can
explain in one sentence) and **Random Forest** (can pick up nonlinear combinations, but costs
interpretability). Simplicity is a feature, not a compromise — I only keep the extra complexity if Random
Forest actually beats Logistic Regression on the same split and metric.

### A leakage check before picking features

Before building anything, I checked what `trend_direction` (my label) is actually made of. The dataset
also has `impressions_last_30d` / `impressions_prev_30d` columns, so I tested whether `trend_pct` is just
the percent change between them:

`(impressions_last_30d - impressions_prev_30d) / impressions_prev_30d * 100`

**Correlation with the real `trend_pct` column: 0.9999999984.** That's not a coincidence — `trend_direction`
and `trend_pct` are computed directly from `*_last_30d` vs `*_prev_30d`. Those six columns
(`impressions_last_30d`, `clicks_last_30d`, `sessions_last_30d`, `impressions_prev_30d`, `clicks_prev_30d`,
`sessions_prev_30d`) are the label in disguise — they can never be features.

That raised a second question: what about `impressions_90d` (and `clicks_90d`, `pageviews_90d`,
`sessions_90d`) — the aggregates I used in my Week-4 baseline? I checked whether they overlap with the
`last_30d` window the label is built from: **`impressions_90d` is always ≥
`impressions_last_30d + impressions_prev_30d`, for 100% of rows** — the 90-day window fully contains the
exact 30 days that define the label. I tested a model with vs. without these four `*_90d` columns: with
them, Precision@20 rose from 0.750 to 0.800 — a real but *modest* lift, not the "suspiciously perfect"
jump a hard leak usually produces. I'm still excluding them, though, on principle rather than just on
that number: they're definitionally entangled with the label's own construction, so any lift they add is
partly circular, not purely predictive.

### Final honest feature set (decision-time only)

`content_age_days`, `days_since_last_update`, `avg_position`, `ctr`, `word_count`, `search_volume`,
`competition`, `content_type` (one-hot) — plus a missingness flag each for `word_count`, `search_volume`,
and `competition`, since roughly a quarter of rows are missing those and "unknown" is itself information
worth keeping, not silently imputing away.

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

pd.set_option("display.width", 120)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)

# --- Leakage check: is trend_pct just last_30d vs prev_30d? ---
computed_trend = (
    (df["impressions_last_30d"] - df["impressions_prev_30d"])
    / df["impressions_prev_30d"].replace(0, np.nan) * 100
)
print("Correlation of computed trend vs actual trend_pct:", round(computed_trend.corr(df["trend_pct"]), 10))

overlap = (df["impressions_90d"] >= df["impressions_last_30d"] + df["impressions_prev_30d"]).mean()
print(f"Share of rows where impressions_90d >= last_30d + prev_30d: {overlap:.1%}")

# --- Final honest feature set ---
honest_num = [
    "content_age_days", "days_since_last_update", "avg_position", "ctr",
    "word_count", "search_volume", "competition",
]
for c in ["word_count", "search_volume", "competition"]:
    df[c + "_missing"] = df[c].isna().astype(int)
    df[c] = df[c].fillna(df[c].median())

honest_features = honest_num + [c + "_missing" for c in ["word_count", "search_volume", "competition"]]
ct_dummies = pd.get_dummies(df["content_type"], prefix="ctype")
X = pd.concat([df[honest_features], ct_dummies], axis=1)
y = df["is_declining_label"]

print(f"\nFinal feature count: {X.shape[1]}")
print(list(X.columns))


Correlation of computed trend vs actual trend_pct: 0.9999999984
Share of rows where impressions_90d >= last_30d + prev_30d: 100.0%

Final feature count: 13
['content_age_days', 'days_since_last_update', 'avg_position', 'ctr', 'word_count', 'search_volume', 'competition', 'word_count_missing', 'search_volume_missing', 'competition_missing', 'ctype_comparison article', 'ctype_feedly article', 'ctype_keyword article']


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Grouped by `client_id`.** I use a `GroupShuffleSplit` (75% train / 25% test) so that no client appears
in both train and test. The real-world use of this model is to score pages for a client's content team —
including clients the model has never seen — so the honest test is "does this generalize to a *new*
client's pages," not "does it generalize to more pages from a client it already trained on." A row-level
random split would let the model quietly memorize client-specific quirks (a particular CMS, a particular
content strategy) and look better than it really is.

I fix `random_state=42` for reproducibility and check that the resulting train/test decline rates are
close enough that the split isn't accidentally skewed.

In [8]:
groups = df["client_id"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, y, groups))

print(f"Train n={len(train_idx)}, Test n={len(test_idx)}")
print(f"Train clients: {groups.iloc[train_idx].nunique()}, Test clients: {groups.iloc[test_idx].nunique()}")
print(f"Train decline rate: {y.iloc[train_idx].mean():.3f}")
print(f"Test decline rate:  {y.iloc[test_idx].mean():.3f}")


Train n=22885, Test n=7115
Train clients: 24, Test clients: 8
Train decline rate: 0.550
Test decline rate:  0.517


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Same data, same test split, same metric (Precision@20 / Precision@50) as everything else this week. I
recompute my Week-4 baseline rule (`stale × visible × impressions_90d`) on the exact same test rows so
it's a fair comparison, not a different sample.

In [9]:
def precision_at_k(scores, y_test, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return y_test.iloc[order].mean()

# --- Week-4 baseline rule, recomputed on the SAME test rows ---
STALE_DAYS, VISIBLE_IMPRESSIONS = 90, 300
stale = (df["days_since_last_update"] >= STALE_DAYS).astype(int)
visible = (df["impressions_90d"] >= VISIBLE_IMPRESSIONS).astype(int)
baseline_score = (stale * visible * df["impressions_90d"]).iloc[test_idx].values

y_test = y.iloc[test_idx]

Xtr, Xte = X.iloc[train_idx].reset_index(drop=True), X.iloc[test_idx].reset_index(drop=True)
ytr = y.iloc[train_idx].reset_index(drop=True)

scaler = StandardScaler()
Xtr_s = pd.DataFrame(scaler.fit_transform(Xtr), columns=X.columns)
Xte_s = pd.DataFrame(scaler.transform(Xte), columns=X.columns)

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(Xtr_s, ytr)
lr_proba = lr.predict_proba(Xte_s)[:, 1]

rf = RandomForestClassifier(n_estimators=300, max_depth=6, class_weight="balanced", random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)
rf_proba = rf.predict_proba(Xte)[:, 1]

comparison = pd.DataFrame([
    {"method": "Base rate (random order)", "precision_at_20": y_test.mean(), "precision_at_50": y_test.mean()},
    {"method": "Week-4 baseline rule",
     "precision_at_20": precision_at_k(baseline_score, y_test, 20),
     "precision_at_50": precision_at_k(baseline_score, y_test, 50)},
    {"method": "Logistic Regression",
     "precision_at_20": precision_at_k(lr_proba, y_test, 20),
     "precision_at_50": precision_at_k(lr_proba, y_test, 50)},
    {"method": "Random Forest",
     "precision_at_20": precision_at_k(rf_proba, y_test, 20),
     "precision_at_50": precision_at_k(rf_proba, y_test, 50)},
])
print(comparison.to_string(index=False))
print("\nWinner: Logistic Regression — beats the baseline and Random Forest at both cutoffs.")


                  method  precision_at_20  precision_at_50
Base rate (random order)         0.516514         0.516514
    Week-4 baseline rule         0.200000         0.300000
     Logistic Regression         0.750000         0.680000
           Random Forest         0.300000         0.440000

Winner: Logistic Regression — beats the baseline and Random Forest at both cutoffs.


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**The baseline rule actually does *worse* than random ordering here (0.200 / 0.300 vs. a 0.517 base
rate).** That's not a bug — it's the Week-4 signal check catching up with me. My baseline ranks purely by
raw impression volume among stale pages, and my own w04 signal audit already showed decline rate is
*not* monotonic with volume (it peaks at moderate/good tiers and drops at the highest tier). Ranking by
volume alone actively points at the wrong pages some of the time. This is a good, honest reason the
learned model matters here, not just a formality.

**Logistic Regression wins clearly (0.750 / 0.680).** Its largest coefficients: `content_age_days`
(negative — younger pages are more likely to be flagged declining, which runs against my Week-4
assumption that "older = needs review"), `days_since_last_update` (positive, but smaller than I expected),
and the missingness flags for `search_volume`/`competition` (rows missing those fields skew toward *not*
declining). Permutation importance (scored on average precision) mostly agrees: `content_age_days` and
`avg_position` matter most, `ctr` next — and `days_since_last_update`, the single signal my whole Week-4
baseline was built on, actually shows a small *negative* importance once the other features are present.
That's a real callback to the w04 signal check, where staleness alone was already a MIXED verdict, not a
clean predictor.

**Random Forest underperforms (0.300 / 0.440) — worse than Logistic Regression, and at Precision@20, worse
than the base rate.** Looking at its top-20 picks: they cluster tightly on one structural profile
(`content_age_days` ≈ 174, `days_since_last_update` ≈ 92–98, `ctr` = 0.0) that Random Forest apparently
learned looks "high risk" from the training clients — but only 6 of those 20 test rows were actually
declining. That profile predicted decline within the training clients but didn't transfer to the held-out
ones — a concrete example of exactly what grouped validation is for, and of the toolkit's own warning:
"does not reward complexity alone."

**Three concrete wrong cases (Logistic Regression's top 20):** all five of LogReg's wrong picks — and
three of its *correct* picks — come from the same client (`client_d029fa3a95`), sharing an almost
identical profile: `content_age_days` = 232, `days_since_last_update` = 183, `ctr` = 0.0, similar position
and word count. The model correctly flags this entire cohort as high-risk, and it's right on most of them
— but it can't tell the winners from the losers *within* that cohort, because on my current feature set
they're nearly indistinguishable. Whatever actually separates the declining pages from the stable/rising
ones in that group isn't captured by content age, staleness, position, CTR, or word count — it's a real
limit of this feature set, not a modeling mistake.


In [10]:

from sklearn.inspection import permutation_importance

print("=== Logistic Regression coefficients (sorted by |coef|) ===")
coefs = pd.Series(lr.coef_[0], index=X.columns).sort_values(key=abs, ascending=False)
print(coefs)

perm = permutation_importance(lr, Xte_s, y_test, scoring="average_precision", n_repeats=20, random_state=42)
print("\n=== Permutation importance (avg-precision drop when shuffled) ===")
print(pd.Series(perm.importances_mean, index=X.columns).sort_values(ascending=False))

print("\n=== Random Forest top-20 test picks: actual trend_direction breakdown ===")
test_df = df.iloc[test_idx].reset_index(drop=True).copy()
test_df["rf_proba"] = rf_proba
test_df["lr_proba"] = lr_proba
rf_top20 = test_df.sort_values("rf_proba", ascending=False).head(20)
print(rf_top20["trend_direction"].value_counts())

print("\n=== Logistic Regression top-20 test picks: wrong cases ===")
lr_top20 = test_df.sort_values("lr_proba", ascending=False).head(20)
wrong = lr_top20[lr_top20["is_declining_label"] == 0]
print(wrong[["content_id", "client_id", "trend_direction", "content_age_days",
             "days_since_last_update", "avg_position", "ctr", "lr_proba"]].to_string(index=False))


=== Logistic Regression coefficients (sorted by |coef|) ===
content_age_days           -0.295168
days_since_last_update      0.221187
search_volume_missing      -0.170834
competition_missing        -0.170834
word_count_missing         -0.133611
ctr                        -0.128662
avg_position               -0.071996
competition                -0.030424
word_count                 -0.026985
ctype_keyword article       0.018259
ctype_feedly article       -0.018259
search_volume               0.003745
ctype_comparison article    0.000000
dtype: float64

=== Permutation importance (avg-precision drop when shuffled) ===
content_age_days            0.024144
avg_position                0.008765
ctr                         0.001289
word_count_missing          0.000608
word_count                  0.000337
ctype_feedly article        0.000000
ctype_comparison article    0.000000
competition_missing        -0.000033
search_volume_missing      -0.000033
search_volume              -0.000038
ctype_k

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.